# MLBB Draft Recommendation Agent
### DQN-Based Hero Draft Assistant for Mobile Legends Bang Bang

**Setup:**
1. Enable GPU: `Runtime → Change runtime type → T4 GPU`
2. Mount Google Drive and place your data files under:
   - `MyDrive/Colab Notebooks/main/heroes_updated.csv`
   - `MyDrive/Colab Notebooks/main/permutated_data2.csv`

## 1. Mount Google Drive

In [57]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE_DIR  = '/content/drive/MyDrive/Colab Notebooks/main'
CHUNK_DIR = os.path.join(BASE_DIR, 'train_chunks')
os.makedirs(CHUNK_DIR, exist_ok=True)

print('Files in BASE_DIR:')
!ls '{BASE_DIR}'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Files in BASE_DIR:
data  mlbb_draft_agentv2.ipynb	Model.ipynb  train_chunks


In [58]:
!ls '/content/drive/MyDrive/Colab Notebooks/main/data'

draft_model_best.pth   heroes_updated.csv
draft_model_final.pth  permutated_data2.csv


## 2. Imports

In [59]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import math
import random
import os
import pickle
from sklearn.model_selection import train_test_split

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')

PyTorch version : 2.10.0+cu128
CUDA available  : True
GPU             : Tesla T4


## 3. Load Data

In [60]:
df               = pd.read_csv(os.path.join('/content/drive/MyDrive/Colab Notebooks/main/data/heroes_updated.csv'))
df_match_history = pd.read_csv(os.path.join('/content/drive/MyDrive/Colab Notebooks/main/data/permutated_data2.csv'))

print(f'Heroes loaded  : {len(df)}')
print(f'Matches loaded : {len(df_match_history)}')
print()
print(df.head())

Heroes loaded  : 131
Matches loaded : 156310

   id     name         roles  specialities        lane possible_lanes  \
0   1     Miya      Marksman   Reap,Damage  Gold Laner            NaN   
1   2  Balmond       Fighter  Damage,Regen     Jungler      Exp Laner   
2   3    Saber      Assassin   Charge,Reap      Roamer        Jungler   
3   4    Alice     Mage,Tank  Charge,Regen   Exp Laner        Jungler   
4   5     Nana  Mage,Support    Poke,Burst   Mid Laner            NaN   

                                                icon  
0  https://static.wikia.nocookie.net/mobile-legen...  
1  https://static.wikia.nocookie.net/mobile-legen...  
2  https://static.wikia.nocookie.net/mobile-legen...  
3  https://static.wikia.nocookie.net/mobile-legen...  
4  https://static.wikia.nocookie.net/mobile-legen...  


## 4. Ban Rate Configuration

Real ban rates scraped from mobilelegends.com/rank (Feb 17, 2026).  
Used to simulate realistic bans during training via weighted random sampling.

In [61]:
BAN_RATES = {
    'Gloo': 81.00,        'Sora': 67.09,        'Aamon': 41.63,
    'Helcurt': 37.93,     'Freya': 35.15,       'Yi Sun-shin': 33.03,
    'Alice': 30.55,       'Estes': 28.16,       'Diggie': 24.59,
    'Saber': 24.27,       'Floryn': 22.42,      'Hayabusa': 21.41,
    'Leomord': 20.30,     'Fredrinn': 20.22,    'Angela': 18.43,
    'Hilda': 18.22,       'Lancelot': 13.79,    'Guinevere': 13.37,
    'Ixia': 13.28,        'Sun': 12.69,         'Yu Zhong': 11.67,
    'Gusion': 11.23,      'Franco': 11.05,      'Granger': 10.72,
    'Grock': 10.12,       'Zetian': 8.80,       'X.Borg': 8.02,
    'Hanzo': 7.40,        'Minotaur': 7.40,     'Belerick': 7.05,
    'Tigreal': 6.85,      'Thamuz': 6.58,       'Minsitthar': 6.47,
    'Fanny': 6.45,        'Kadita': 6.15,       'Eudora': 6.02,
    'Lesley': 5.94,       'Lapu-Lapu': 5.45,    'Kalea': 5.01,
    'Cici': 4.97,         'Rafaela': 4.90,      'Chip': 4.61,
    'Hanabi': 4.57,       'Nana': 4.34,         'Yin': 4.29,
    'Zhuxin': 4.18,       'Claude': 4.16,       'Karrie': 4.12,
    'Harley': 3.50,       'Atlas': 3.37,        'Johnson': 3.29,
    'Obsidia': 3.29,      'Julian': 3.17,       'Chou': 2.85,
    'Miya': 2.68,         'Esmeralda': 2.48,    'Cyclops': 2.44,
    'Natalia': 2.38,      'Lolita': 2.25,       'Alucard': 2.21,
    'Akai': 2.20,         'Karina': 2.05,       'Joy': 1.86,
    'Argus': 1.84,        'Lukas': 1.80,        'Uranus': 1.72,
    'Vexana': 1.71,       'Silvanna': 1.67,     'Badang': 1.66,
    'Khufra': 1.64,       'Carmilla': 1.50,     'Phoveus': 1.47,
    'Arlott': 1.45,       'Alpha': 1.43,        'Pharsa': 1.39,
    'Layla': 1.36,        'Selena': 1.33,       'Benedetta': 1.29,
    'Suyou': 1.17,        'Kaja': 1.16,         'Clint': 1.16,
    'Valir': 1.11,        'Hylos': 1.07,        'Kagura': 1.03,
    'Gatotkaca': 1.02,    'Melissa': 0.99,      "Chang'e": 0.95,
    'Mathilda': 0.95,     'Kimmy': 0.91,        'Lylia': 0.90,
    'Ruby': 0.80,         'Zilong': 0.75,       'Faramis': 0.73,
    'Wanwan': 0.71,       'Irithel': 0.66,      'Odette': 0.65,
    'Martis': 0.57,       'Aldous': 0.56,       'Cecilion': 0.54,
    'Dyrroth': 0.54,      'Valentina': 0.53,    'Nolan': 0.52,
    'Lunox': 0.51,        'Khaleed': 0.50,      'Ling': 0.46,
    'Brody': 0.46,        'Xavier': 0.44,       'Natan': 0.42,
    'Jawhead': 0.37,      'Popol and Kupa': 0.37, 'Paquito': 0.35,
    'Terizla': 0.35,      'Gord': 0.34,         'Yve': 0.33,
    'Bane': 0.33,         'Masha': 0.32,        'Zhask': 0.32,
    'Balmond': 0.32,      'Baxia': 0.30,        'Aurora': 0.30,
    'Vale': 0.29,         'Beatrix': 0.29,      'Moskov': 0.27,
    'Aulus': 0.23,        'Novaria': 0.22,      'Roger': 0.17,
    'Barats': 0.14,       'Luo Yi': 0.14,       'Edith': 0.14,
    'Bruno': 0.11,        'Harith': 0.11,
}

TEMPERATURE = 20.0  # Higher = more uniform; lower = top heroes dominate

def _compute_softmax_weights(ban_rates, temperature):
    heroes = list(ban_rates.keys())
    rates  = [ban_rates[h] for h in heroes]
    scaled = [r / temperature for r in rates]
    max_s  = max(scaled)
    exps   = [math.exp(s - max_s) for s in scaled]
    total  = sum(exps)
    return [e / total for e in exps]

# Only heroes present in both ban_rates AND hero_pool are eligible for banning
HERO_POOL       = list({row['name']: row['id'] - 1 for _, row in df.iterrows()})
SOFTMAX_WEIGHTS = _compute_softmax_weights(BAN_RATES, TEMPERATURE)

def getBannedHeroes():
    keys = [(random.random() ** (1.0 / w), hero)
            for hero, w in zip(HERO_POOL, SOFTMAX_WEIGHTS)]
    keys.sort(reverse=True)
    return [hero for _, hero in keys[:10]]

print('Sample ban list:', getBannedHeroes())

Sample ban list: ['Miya', 'Clint', 'Phoveus', 'Yin', 'Nana', 'Balmond', 'Hanabi', 'Aurora', 'Dyrroth', 'Estes']


## 5. Hero Metadata & Encoding

Each hero is encoded as a multi-hot vector covering **roles**, **specialities**, and **lanes** (primary + flex).

In [62]:
# Attribute parsing helpers
def split_attr(val):
    """Splits comma- or slash-separated attribute strings into a clean list."""
    if pd.isna(val):
        return []
    return [x.strip() for x in str(val).replace('/', ',').split(',')]

def clean_role(r):
    r = r.strip()
    if r == 'Supprot': return 'Support'  # typo fix
    if r == 'Jungle':  return None       # invalid entry
    return r

def clean_lane(val):
    lane = str(val).strip()
    return 'Exp Laner' if lane == 'EXP Laner' else lane

def parse_possible_lanes(val):
    if pd.isna(val):
        return []
    val = str(val).strip()
    if val.lower() == 'none':
        return []
    return [clean_lane(l) for l in val.replace('/', ',').split(',') if clean_lane(l.strip())]

# Fix hero name inconsistencies in match history
pick_cols = [f'winpick{i}' for i in range(1, 6)] + [f'losepick{i}' for i in range(1, 6)]
for col in pick_cols:
    df_match_history[col] = df_match_history[col].str.strip().replace('Change', "Chang'e")

# Hero lookup
hero_to_id = {row['name'].strip(): int(row['id']) - 1 for _, row in df.iterrows()}
id_to_hero = {v: k for k, v in hero_to_id.items()}
NUM_HEROES = len(hero_to_id)

# Verify no unknown heroes in match data
all_match_heroes = set()
for col in pick_cols:
    all_match_heroes.update(df_match_history[col].unique())
missing = all_match_heroes - set(df['name'].str.strip())
print(f'Unknown heroes in match data: {missing}  ← should be empty')

# Per-hero attribute dicts
hero_roles = {}
hero_specs = {}
hero_lanes = {}

for _, row in df.iterrows():
    name             = row['name'].strip()
    hero_roles[name] = [clean_role(r) for r in split_attr(row['roles']) if clean_role(r)]
    hero_specs[name] = split_attr(row['specialities'])
    hero_lanes[name] = list(set([clean_lane(row['lane'])] + parse_possible_lanes(row['possible_lanes'])))

# Encoding categories
ROLES = sorted(['Assassin', 'Fighter', 'Mage', 'Marksman', 'Support', 'Tank'])
SPECS = sorted(['Burst', 'Charge', 'Chase', 'Control', 'Crowd Control', 'Damage',
                'Finisher', 'Guard', 'Initiator', 'Magic Damage', 'Mixed Damage',
                'Poke', 'Push', 'Reap', 'Regen', 'Support'])
LANES = sorted(['Exp Laner', 'Gold Laner', 'Jungler', 'Mid Laner', 'Roamer'])

role_to_id = {r: i for i, r in enumerate(ROLES)}
spec_to_id = {s: i for i, s in enumerate(SPECS)}
lane_to_id = {l: i for i, l in enumerate(LANES)}

NUM_ROLES = len(ROLES)   # 6
NUM_SPECS = len(SPECS)   # 16
NUM_LANES = len(LANES)   # 5
STATE_DIM = NUM_HEROES * 2 + NUM_ROLES * 2 + NUM_SPECS * 2 + NUM_LANES * 2  # 316

print(f'Heroes: {NUM_HEROES} | Roles: {NUM_ROLES} | Specs: {NUM_SPECS} | Lanes: {NUM_LANES}')
print(f'State vector size: {STATE_DIM}')
print()
print('Flex lane spot check:')
print(f'  Balmond : {hero_lanes["Balmond"]}  ← should include Jungler + Exp Laner')
print(f'  Saber   : {hero_lanes["Saber"]}    ← should include Roamer + Jungler')
print(f'  Miya    : {hero_lanes["Miya"]}     ← Gold Laner only')

Unknown heroes in match data: set()  ← should be empty
Heroes: 131 | Roles: 6 | Specs: 16 | Lanes: 5
State vector size: 316

Flex lane spot check:
  Balmond : ['Jungler', 'Exp Laner']  ← should include Jungler + Exp Laner
  Saber   : ['Jungler', 'Roamer']    ← should include Roamer + Jungler
  Miya    : ['Gold Laner']     ← Gold Laner only


## 6. State Encoding

| Segment | Size | Description |
|---|---|---|
| Ally heroes | 131 | One-hot: ally picks |
| Enemy heroes | 131 | One-hot: enemy picks |
| Ally roles | 6 | Multi-hot: roles in ally team |
| Enemy roles | 6 | Multi-hot: roles in enemy team |
| Ally specs | 16 | Multi-hot: specialities in ally team |
| Enemy specs | 16 | Multi-hot: specialities in enemy team |
| Ally lanes | 5 | Multi-hot: lanes covered by ally |
| Enemy lanes | 5 | Multi-hot: lanes covered by enemy |
| **Total** | **316** | |

In [63]:
def encode_team(heroes):
    hero_vec = np.zeros(NUM_HEROES, dtype=np.float32)
    role_vec = np.zeros(NUM_ROLES,  dtype=np.float32)
    spec_vec = np.zeros(NUM_SPECS,  dtype=np.float32)
    lane_vec = np.zeros(NUM_LANES,  dtype=np.float32)
    for h in heroes:
        hero_vec[hero_to_id[h]] = 1.0
        for r in hero_roles.get(h, []):
            if r in role_to_id: role_vec[role_to_id[r]] = 1.0
        for s in hero_specs.get(h, []):
            if s in spec_to_id: spec_vec[spec_to_id[s]] = 1.0
        for l in hero_lanes.get(h, []):
            if l in lane_to_id: lane_vec[lane_to_id[l]] = 1.0
    return hero_vec, role_vec, spec_vec, lane_vec


def encode_state(ally_picks, enemy_picks):
    ah, ar, as_, al = encode_team(ally_picks)
    eh, er, es, el  = encode_team(enemy_picks)
    state = np.concatenate([ah, eh, ar, er, as_, es, al, el])
    return torch.tensor(state, dtype=torch.float32)


# Sanity check
state = encode_state(['Yu Zhong', 'Fredrinn', 'Karrie'], ['Fanny', 'Gusion', 'Lesley'])
print(f'State shape: {state.shape}  ← should be torch.Size([316])')
print('\nAlly  heroes :', [id_to_hero[i] for i in torch.where(state[0:131]   == 1)[0].tolist()])
print('Enemy heroes :', [id_to_hero[i] for i in torch.where(state[131:262] == 1)[0].tolist()])
print('Ally  roles  :', [ROLES[i]      for i in torch.where(state[262:268] == 1)[0].tolist()])
print('Enemy roles  :', [ROLES[i]      for i in torch.where(state[268:274] == 1)[0].tolist()])
print('Ally  lanes  :', [LANES[i]      for i in torch.where(state[306:311] == 1)[0].tolist()])
print('Enemy lanes  :', [LANES[i]      for i in torch.where(state[311:316] == 1)[0].tolist()])

State shape: torch.Size([316])  ← should be torch.Size([316])

Ally  heroes : ['Karrie', 'Yu Zhong', 'Fredrinn']
Enemy heroes : ['Fanny', 'Lesley', 'Gusion']
Ally  roles  : ['Fighter', 'Marksman', 'Tank']
Enemy roles  : ['Assassin', 'Marksman']
Ally  lanes  : ['Exp Laner', 'Gold Laner', 'Jungler']
Enemy lanes  : ['Gold Laner', 'Jungler', 'Mid Laner']


## 7. Draft Environment

Simulates a 10-pick MLBB draft from one team's perspective.

**Pick order:** `[0, 1, 1, 0, 0, 1, 1, 0, 0, 1]`

**Shaped reward:** small bonus at every step for building a balanced composition:
- Each new **role** added → `+0.05`
- Each new **lane** covered → `+0.05`
- Final step win → `+1.0` | loss → `-1.0`

In [64]:
PICK_ORDER = [0, 1, 1, 0, 0, 1, 1, 0, 0, 1]


def compute_shaped_reward(prev_ally, new_ally, is_done, did_win):
    if is_done:
        return 1.0 if did_win else -1.0
    prev_roles = set(r for h in prev_ally for r in hero_roles.get(h, []))
    new_roles  = set(r for h in new_ally  for r in hero_roles.get(h, []))
    prev_lanes = set(l for h in prev_ally for l in hero_lanes.get(h, []))
    new_lanes  = set(l for h in new_ally  for l in hero_lanes.get(h, []))
    return 0.05 * len(new_roles - prev_roles) + 0.05 * len(new_lanes - prev_lanes)


class MLBBDraftEnv:
    """
    Simulates one MLBB draft episode from a single team's perspective.

    Args:
        win_heroes  : list[str] — winning team's picks in order
        lose_heroes : list[str] — losing team's picks in order
        fp_is_win   : bool — whether the first-picking team won
        perspective : int — which team we learn from (0 or 1)
    """
    def __init__(self, win_heroes, lose_heroes, fp_is_win, perspective=0):
        if fp_is_win:
            self.team     = {0: win_heroes,  1: lose_heroes}
            self.team_won = {0: True,        1: False}
        else:
            self.team     = {0: lose_heroes, 1: win_heroes}
            self.team_won = {0: False,       1: True}
        self.perspective = perspective
        self.reset()

    def reset(self):
        self.step_idx = 0
        self.picks    = {0: [], 1: []}
        return self._get_state()

    def _get_state(self):
        return encode_state(self.picks[self.perspective], self.picks[1 - self.perspective])

    def step(self, hero):
        assert self.step_idx < 10, 'Episode already finished.'
        picker    = PICK_ORDER[self.step_idx]
        prev_ally = list(self.picks[self.perspective])
        self.picks[picker].append(hero)
        self.step_idx += 1
        done   = (self.step_idx == 10)
        action = hero_to_id[hero]
        state  = encode_state(
            self.picks[self.perspective] if picker != self.perspective else prev_ally,
            self.picks[1 - self.perspective]
        )
        next_state = self._get_state()
        reward = (
            compute_shaped_reward(prev_ally, self.picks[self.perspective], done, self.team_won[self.perspective])
            if picker == self.perspective else 0.0
        )
        return state, action, reward, next_state, done

    def run_episode(self):
        self.reset()
        transitions = []
        pick_counts = {0: 0, 1: 0}
        for step in range(10):
            picker = PICK_ORDER[step]
            hero   = self.team[picker][pick_counts[picker]]
            pick_counts[picker] += 1
            s, a, r, ns, done = self.step(hero)
            if picker == self.perspective:
                transitions.append((s, a, r, ns, done))
        return transitions


def build_transitions(df_matches, both_perspectives=True):
    all_transitions, skipped = [], 0
    for _, row in df_matches.iterrows():
        win_heroes  = [row[f'winpick{i}'].strip()  for i in range(1, 6)]
        lose_heroes = [row[f'losepick{i}'].strip() for i in range(1, 6)]
        if any(h not in hero_to_id for h in win_heroes + lose_heroes):
            skipped += 1
            continue
        fp_is_win    = random.random() < 0.5
        perspectives = [0, 1] if both_perspectives else [random.randint(0, 1)]
        for pov in perspectives:
            env = MLBBDraftEnv(win_heroes, lose_heroes, fp_is_win, perspective=pov)
            all_transitions.extend(env.run_episode())
    print(f'Processed: {len(df_matches) - skipped} | Skipped: {skipped} | Transitions: {len(all_transitions)}')
    return all_transitions


def build_and_save_transitions_chunked(df_matches, save_dir, both_perspectives=True, chunk_size=100_000):
    """Saves transitions to disk in chunks to avoid RAM exhaustion."""
    os.makedirs(save_dir, exist_ok=True)
    buffer, chunk_idx, skipped, total = [], 0, 0, 0

    def flush():
        nonlocal chunk_idx
        path = os.path.join(save_dir, f'chunk_{chunk_idx:04d}.pkl')
        with open(path, 'wb') as f:
            pickle.dump(buffer, f)
        print(f'  Saved chunk {chunk_idx} → {len(buffer):,} transitions ({path})')
        chunk_idx += 1
        buffer.clear()

    for _, row in df_matches.iterrows():
        win_heroes  = [row[f'winpick{i}'].strip()  for i in range(1, 6)]
        lose_heroes = [row[f'losepick{i}'].strip() for i in range(1, 6)]
        if any(h not in hero_to_id for h in win_heroes + lose_heroes):
            skipped += 1
            continue
        fp_is_win    = random.random() < 0.5
        perspectives = [0, 1] if both_perspectives else [random.randint(0, 1)]
        for pov in perspectives:
            env = MLBBDraftEnv(win_heroes, lose_heroes, fp_is_win, perspective=pov)
            buffer.extend(env.run_episode())
        if len(buffer) >= chunk_size:
            total += len(buffer)
            flush()

    if buffer:
        total += len(buffer)
        flush()

    print(f'\nDone. Transitions: {total:,} | Chunks: {chunk_idx} | Skipped: {skipped}')
    return chunk_idx

## 8. Replay Memory & DQN Model

```
Input (316) → Linear(256) → ReLU → Dropout(0.4)
            → Linear(128) → ReLU → Dropout(0.4)
            → Output (131 Q-values, one per hero)
```

In [65]:
class ReplayMemory:
    def __init__(self, capacity):
        self.capacity = capacity
        self.memory   = []
        self.position = 0

    def insert(self, state, action, reward, next_state, done):
        transition = (
            state.unsqueeze(0),
            torch.tensor([[action]],  dtype=torch.long),
            torch.tensor([[reward]],  dtype=torch.float32),
            next_state.unsqueeze(0),
            torch.tensor([[done]],    dtype=torch.bool),
        )
        if len(self.memory) < self.capacity:
            self.memory.append(None)
        self.memory[self.position] = transition
        self.position = (self.position + 1) % self.capacity

    def sample(self, batch_size):
        batch = random.sample(self.memory, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (torch.cat(states), torch.cat(actions), torch.cat(rewards),
                torch.cat(next_states), torch.cat(dones))

    def can_sample(self, batch_size):
        return len(self.memory) >= batch_size * 10

    def __len__(self):
        return len(self.memory)


class DQN(nn.Module):
    def __init__(self, input_dim=316, output_dim=131):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256, 128),       nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(128, output_dim)
        )

    def forward(self, x):
        return self.net(x)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device    : {device}')
print(f'State dim : {STATE_DIM} | Heroes: {NUM_HEROES}')

Device    : cuda
State dim : 316 | Heroes: 131


## 9. Evaluation — Recall@K

Measures: *"Of the heroes the winning team actually picked, how many appear in the model's top-K recommendations?"*

| Recall@5 | Recall@10 | Signal |
|---|---|---|
| ~3.8% | ~7.6% | Random baseline |
| 8–12% | 15–20% | Weak |
| 15–20% | 25–35% | Good |
| 25%+ | 40%+ | Strong |

In [66]:
def recall_at_k(model, df_matches, k_values=(5, 10), n_samples=500):
    model.eval()
    sample_df    = df_matches.sample(min(n_samples, len(df_matches)), random_state=42)
    win_recalls  = {k: [] for k in k_values}
    lose_recalls = {k: [] for k in k_values}

    with torch.no_grad():
        for _, row in sample_df.iterrows():
            win_heroes  = [row[f'winpick{i}'].strip()  for i in range(1, 6)]
            lose_heroes = [row[f'losepick{i}'].strip() for i in range(1, 6)]
            if any(h not in hero_to_id for h in win_heroes + lose_heroes):
                continue

            for our_team, their_team, recall_dict in [
                (win_heroes,  lose_heroes, win_recalls),
                (lose_heroes, win_heroes,  lose_recalls),
            ]:
                ally_so_far, enemy_so_far = [], []
                pick_counts = {0: 0, 1: 0}
                for picker in PICK_ORDER:
                    if picker == 0:
                        actual_pick = our_team[pick_counts[0]]
                        state       = encode_state(ally_so_far, enemy_so_far).to(device)
                        q_vals      = model(state.unsqueeze(0)).squeeze(0)
                        taken       = set(ally_so_far + enemy_so_far)
                        scored      = sorted(
                            [(h, q_vals[hero_to_id[h]].item()) for h in hero_to_id if h not in taken],
                            key=lambda x: x[1], reverse=True
                        )
                        for k in k_values:
                            recall_dict[k].append(1.0 if actual_pick in [h for h, _ in scored[:k]] else 0.0)
                        ally_so_far.append(our_team[pick_counts[0]])
                        pick_counts[0] += 1
                    else:
                        enemy_so_far.append(their_team[pick_counts[1]])
                        pick_counts[1] += 1

    return (
        tuple(np.mean(win_recalls[k])  * 100 for k in k_values) +
        tuple(np.mean(lose_recalls[k]) * 100 for k in k_values)
    )

## 10. Training

| Hyperparameter | Value | Reason |
|---|---|---|
| `GAMMA` | 0.99 | Future rewards matter in a 5-step draft |
| `BATCH_SIZE` | 256 | Stable gradients |
| `LR` | 1e-4 | Conservative for small dataset |
| `weight_decay` | 1e-4 | L2 regularization |
| `TARGET_SYNC_EVERY` | 10 | Stable target network |
| `MAX_PATIENCE` | 15 | Early stopping (checked every 5 epochs) |

In [67]:
# Hyperparameters
GAMMA             = 0.99
BATCH_SIZE        = 256
NUM_EPOCHS        = 500
TARGET_SYNC_EVERY = 10
LR                = 1e-4
MAX_PATIENCE      = 15
CHUNK_SIZE        = 100_000

MODEL_BEST  = os.path.join('/content/drive/MyDrive/Colab Notebooks/main/data/draft_model_best.pth')
MODEL_FINAL = os.path.join('/content/drive/MyDrive/Colab Notebooks/main/data/draft_model_final.pth')

# Train / Val / Test split
train_df, temp_df = train_test_split(df_match_history, test_size=0.30, random_state=42)
val_df,   test_df = train_test_split(temp_df,          test_size=0.50, random_state=42)
print(f'Split — Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)} matches')

# Build / load train chunks
existing_chunks = [f for f in os.listdir(CHUNK_DIR) if f.endswith('.pkl')] if os.path.exists(CHUNK_DIR) else []
if not existing_chunks:
    print('\nBuilding train transitions and saving to Drive...')
    num_chunks = build_and_save_transitions_chunked(train_df, CHUNK_DIR, chunk_size=CHUNK_SIZE)
else:
    num_chunks = len(existing_chunks)
    print(f'\nFound {num_chunks} existing chunks in Drive — skipping build.')

# Val transitions (small enough for RAM)
print('\nBuilding val transitions...')
val_transitions = build_transitions(val_df, both_perspectives=True)

Split — Train: 109417 | Val: 23446 | Test: 23447 matches

Found 11 existing chunks in Drive — skipping build.

Building val transitions...
Processed: 23446 | Skipped: 0 | Transitions: 234460


In [68]:
def train_chunked(val_transitions, chunk_dir, num_chunks,
                  epochs_per_chunk=50, batch_size=BATCH_SIZE):
    """Train by rotating through on-disk chunks, spending epochs_per_chunk epochs on each."""

    # Validation replay buffer
    val_memory = ReplayMemory(capacity=20_000)
    for s, a, r, ns, done in val_transitions:
        val_memory.insert(s, a, r, ns, done)

    print(f'Val buffer: {len(val_memory):,}')

    # Networks
    policy_net = DQN(STATE_DIM, NUM_HEROES).to(device)
    target_net = DQN(STATE_DIM, NUM_HEROES).to(device)

    target_net.load_state_dict(policy_net.state_dict())
    target_net.eval()

    optimizer = optim.Adam(policy_net.parameters(), lr=LR, weight_decay=1e-4)
    criterion = nn.MSELoss()

    best_val_loss = float('inf')

    # Locate chunk files
    chunk_files = sorted([
        os.path.join(chunk_dir, f)
        for f in os.listdir(chunk_dir)
        if f.endswith('.pkl')
    ])

    print(f'Found {len(chunk_files)} chunk files')

    header = f"{'Epoch':>6} | {'Chunk':>14} | {'Train Loss':>10} | {'Val Loss':>9} | "\
             f"{'Win R@5':>7} | {'Win R@10':>8} | {'Lose R@5':>8} | {'Lose R@10':>9}"

    print(f'\n{header}')
    print('─' * 90)

    global_epoch = 0

    # Iterate through chunks
    for chunk_idx, chunk_path in enumerate(chunk_files):

        chunk_name = os.path.basename(chunk_path)
        print(f'\n── Loading chunk {chunk_idx+1}/{len(chunk_files)}: {chunk_name} ──')

        with open(chunk_path, 'rb') as f:
            chunk_data = pickle.load(f)

        train_memory = ReplayMemory(capacity=len(chunk_data))

        for s, a, r, ns, done in chunk_data:
            train_memory.insert(s, a, r, ns, done)

        del chunk_data

        if not train_memory.can_sample(batch_size):
            print(f'Chunk {chunk_name} too small, skipping.')
            continue

        steps = max(len(train_memory) // batch_size, 1)

        # Train epochs for this chunk
        for local_epoch in range(epochs_per_chunk):

            policy_net.train()
            epoch_loss = 0.0

            for _ in range(steps):

                states, actions, rewards, next_states, dones = train_memory.sample(batch_size)

                states = states.to(device)
                actions = actions.to(device)
                rewards = rewards.to(device)
                next_states = next_states.to(device)
                dones = dones.to(device)

                # Current Q values
                q_values = policy_net(states).gather(1, actions)

                # Target Q values
                with torch.no_grad():
                    max_next_q = target_net(next_states).max(dim=1, keepdim=True).values
                    target_q = rewards + GAMMA * max_next_q * (~dones).float()

                loss = criterion(q_values, target_q)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                epoch_loss += loss.item()

            avg_loss = epoch_loss / steps

            # Sync target network
            if global_epoch % TARGET_SYNC_EVERY == 0:
                target_net.load_state_dict(policy_net.state_dict())

            # Validation every 5 epochs
            if True:

                policy_net.eval()
                val_losses = []

                with torch.no_grad():

                    for _ in range(10):

                        vs, va, vr, vns, vd = val_memory.sample(batch_size)

                        vs = vs.to(device)
                        va = va.to(device)
                        vr = vr.to(device)
                        vns = vns.to(device)
                        vd = vd.to(device)

                        vq = policy_net(vs).gather(1, va)

                        vt = vr + GAMMA * target_net(vns).max(dim=1, keepdim=True).values * (~vd).float()

                        val_losses.append(criterion(vq, vt).item())

                val_loss = np.mean(val_losses)

                wr5, wr10, lr5, lr10 = recall_at_k(policy_net, val_df, k_values=[5, 10])

                print(f'{global_epoch:>6} | {chunk_name:>14} | {avg_loss:>10.4f} | {val_loss:>9.4f} | '
                      f'{wr5:>6.1f}% | {wr10:>7.1f}% | {lr5:>7.1f}% | {lr10:>8.1f}%')

                # Save best model (no early stopping)
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    torch.save(policy_net.state_dict(), MODEL_BEST)
                    print(f'  ✓ Best model saved (val_loss={best_val_loss:.4f})')

                policy_net.train()

            global_epoch += 1

    # Save final model
    torch.save(policy_net.state_dict(), MODEL_FINAL)

    print(f'\nSaved → Best  : {MODEL_BEST}')
    print(f'Saved → Final : {MODEL_FINAL}')

    return policy_net, target_net


policy_net, target_net = train_chunked(val_transitions, CHUNK_DIR, num_chunks)

Val buffer: 20,000
Found 11 chunk files

 Epoch |          Chunk | Train Loss |  Val Loss | Win R@5 | Win R@10 | Lose R@5 | Lose R@10
──────────────────────────────────────────────────────────────────────────────────────────

── Loading chunk 1/11: chunk_0000.pkl ──
     0 | chunk_0000.pkl |     0.1309 |    0.1517 |    4.8% |     9.6% |     4.2% |      8.4%
  ✓ Best model saved (val_loss=0.1517)
     1 | chunk_0000.pkl |     0.1273 |    0.1161 |    4.2% |     9.4% |     2.9% |      7.6%
  ✓ Best model saved (val_loss=0.1161)
     2 | chunk_0000.pkl |     0.1134 |    0.1078 |    4.4% |     9.4% |     3.2% |      8.8%
  ✓ Best model saved (val_loss=0.1078)
     3 | chunk_0000.pkl |     0.1110 |    0.0984 |    5.3% |     9.4% |     4.0% |      8.5%
  ✓ Best model saved (val_loss=0.0984)
     4 | chunk_0000.pkl |     0.1120 |    0.1139 |    4.3% |     9.4% |     3.9% |      8.0%
     5 | chunk_0000.pkl |     0.1104 |    0.1038 |    4.5% |     8.9% |     3.6% |      8.4%
     6 | chunk_0000

In [69]:
# def train_chunked(val_transitions, chunk_dir, num_chunks,
#                   epochs_per_chunk=50, batch_size=BATCH_SIZE):  # <-- replaced num_epochs with epochs_per_chunk
#     """Train by rotating through on-disk chunks, spending epochs_per_chunk epochs on each."""
#     val_memory = ReplayMemory(capacity=20_000)
#     for s, a, r, ns, done in val_transitions:
#         val_memory.insert(s, a, r, ns, done)
#     print(f'Val buffer: {len(val_memory):,}')

#     policy_net = DQN(STATE_DIM, NUM_HEROES).to(device)
#     target_net = DQN(STATE_DIM, NUM_HEROES).to(device)
#     target_net.load_state_dict(policy_net.state_dict())
#     target_net.eval()

#     optimizer     = optim.Adam(policy_net.parameters(), lr=LR, weight_decay=1e-4)
#     criterion     = nn.MSELoss()
#     best_val_loss = float('inf')
#     patience      = 0

#     chunk_files = sorted([os.path.join(chunk_dir, f)
#                           for f in os.listdir(chunk_dir) if f.endswith('.pkl')])
#     print(f'Found {len(chunk_files)} chunk files')

#     header = f"{'Epoch':>6} | {'Chunk':>14} | {'Train Loss':>10} | {'Val Loss':>9} | "\
#              f"{'Win R@5':>7} | {'Win R@10':>8} | {'Lose R@5':>8} | {'Lose R@10':>9}"
#     print(f'\n{header}')
#     print('─' * 90)

#     global_epoch = 0
#     early_stop   = False

#     for chunk_idx, chunk_path in enumerate(chunk_files):  # <-- outer loop: each chunk
#         if early_stop:
#             break

#         chunk_name = os.path.basename(chunk_path)
#         print(f'\n── Loading chunk {chunk_idx+1}/{len(chunk_files)}: {chunk_name} ──')

#         with open(chunk_path, 'rb') as f:
#             chunk_data = pickle.load(f)

#         train_memory = ReplayMemory(capacity=len(chunk_data))
#         for s, a, r, ns, done in chunk_data:
#             train_memory.insert(s, a, r, ns, done)
#         del chunk_data

#         if not train_memory.can_sample(batch_size):
#             print(f'Chunk {chunk_name} too small, skipping.')
#             continue

#         steps = max(len(train_memory) // batch_size, 1)

#         for local_epoch in range(epochs_per_chunk):  # <-- inner loop: 50 epochs per chunk
#             policy_net.train()
#             epoch_loss = 0.0
#             for _ in range(steps):
#                 states, actions, rewards, next_states, dones = train_memory.sample(batch_size)
#                 states, actions, rewards, next_states, dones = (
#                     states.to(device), actions.to(device), rewards.to(device),
#                     next_states.to(device), dones.to(device)
#                 )
#                 q_values = policy_net(states).gather(1, actions)
#                 with torch.no_grad():
#                     max_next_q = target_net(next_states).max(dim=1, keepdim=True).values
#                     target_q   = rewards + GAMMA * max_next_q * (~dones).float()
#                 loss = criterion(q_values, target_q)
#                 optimizer.zero_grad()
#                 loss.backward()
#                 optimizer.step()
#                 epoch_loss += loss.item()

#             avg_loss = epoch_loss / steps

#             if global_epoch % TARGET_SYNC_EVERY == 0:
#                 target_net.load_state_dict(policy_net.state_dict())

#             # Evaluate every 5 local epochs
#             if local_epoch % 5 == 0:
#                 policy_net.eval()
#                 val_losses = []
#                 with torch.no_grad():
#                     for _ in range(10):
#                         vs, va, vr, vns, vd = val_memory.sample(batch_size)
#                         vs, va, vr, vns, vd = (vs.to(device), va.to(device), vr.to(device),
#                                                 vns.to(device), vd.to(device))
#                         vq = policy_net(vs).gather(1, va)
#                         vt = vr + GAMMA * target_net(vns).max(dim=1, keepdim=True).values * (~vd).float()
#                         val_losses.append(criterion(vq, vt).item())
#                 val_loss = np.mean(val_losses)
#                 wr5, wr10, lr5, lr10 = recall_at_k(policy_net, val_df, k_values=[5, 10])

#                 print(f'{global_epoch:>6} | {chunk_name:>14} | {avg_loss:>10.4f} | {val_loss:>9.4f} | '
#                       f'{wr5:>6.1f}% | {wr10:>7.1f}% | {lr5:>7.1f}% | {lr10:>8.1f}%')

#                 if val_loss < best_val_loss:
#                     best_val_loss = val_loss
#                     patience      = 0
#                     torch.save(policy_net.state_dict(), MODEL_BEST)
#                     print(f'  ✓ Best model saved (val_loss={best_val_loss:.4f})')
#                 else:
#                     patience += 1
#                     if patience >= MAX_PATIENCE:
#                         print(f'\nEarly stopping at global_epoch {global_epoch} — best val_loss={best_val_loss:.4f}')
#                         early_stop = True
#                         break

#                 policy_net.train()

#             global_epoch += 1

#     torch.save(policy_net.state_dict(), MODEL_FINAL)
#     print(f'\nSaved → Best  : {MODEL_BEST}')
#     print(f'Saved → Final : {MODEL_FINAL}')
#     return policy_net, target_net

# policy_net, target_net = train_chunked(val_transitions, CHUNK_DIR, num_chunks)

In [70]:
# def train_chunked(val_transitions, chunk_dir, num_chunks,
#                   num_epochs=NUM_EPOCHS, batch_size=BATCH_SIZE):
#     """Train by rotating through on-disk chunks to stay within Colab RAM limits."""
#     # Val buffer
#     val_memory = ReplayMemory(capacity=20_000)
#     for s, a, r, ns, done in val_transitions:
#         val_memory.insert(s, a, r, ns, done)
#     print(f'Val buffer: {len(val_memory):,}')

#     # Networks
#     policy_net = DQN(STATE_DIM, NUM_HEROES).to(device)
#     target_net = DQN(STATE_DIM, NUM_HEROES).to(device)
#     target_net.load_state_dict(policy_net.state_dict())
#     target_net.eval()

#     optimizer     = optim.Adam(policy_net.parameters(), lr=LR, weight_decay=1e-4)
#     criterion     = nn.MSELoss()
#     best_val_loss = float('inf')
#     patience      = 0

#     chunk_files = sorted([os.path.join(chunk_dir, f)
#                           for f in os.listdir(chunk_dir) if f.endswith('.pkl')])
#     print(f'Found {len(chunk_files)} chunk files')

#     header = f"{'Epoch':>6} | {'Chunk':>14} | {'Train Loss':>10} | {'Val Loss':>9} | "\
#              f"{'Win R@5':>7} | {'Win R@10':>8} | {'Lose R@5':>8} | {'Lose R@10':>9}"
#     print(f'\n{header}')
#     print('─' * 90)

#     for epoch in range(num_epochs):
#         chunk_path = chunk_files[epoch % len(chunk_files)]
#         chunk_name = os.path.basename(chunk_path)

#         with open(chunk_path, 'rb') as f:
#             chunk_data = pickle.load(f)

#         train_memory = ReplayMemory(capacity=len(chunk_data))
#         for s, a, r, ns, done in chunk_data:
#             train_memory.insert(s, a, r, ns, done)
#         del chunk_data

#         if not train_memory.can_sample(batch_size):
#             print(f'Epoch {epoch}: chunk too small, skipping.')
#             continue

#         steps = max(len(train_memory) // batch_size, 1)

#         # Training
#         policy_net.train()
#         epoch_loss = 0.0
#         for _ in range(steps):
#             states, actions, rewards, next_states, dones = train_memory.sample(batch_size)
#             states, actions, rewards, next_states, dones = (
#                 states.to(device), actions.to(device), rewards.to(device),
#                 next_states.to(device), dones.to(device)
#             )
#             q_values = policy_net(states).gather(1, actions)
#             with torch.no_grad():
#                 max_next_q = target_net(next_states).max(dim=1, keepdim=True).values
#                 target_q   = rewards + GAMMA * max_next_q * (~dones).float()
#             loss = criterion(q_values, target_q)
#             optimizer.zero_grad()
#             loss.backward()
#             optimizer.step()
#             epoch_loss += loss.item()

#         avg_loss = epoch_loss / steps

#         if epoch % TARGET_SYNC_EVERY == 0:
#             target_net.load_state_dict(policy_net.state_dict())

#         # Evaluate every 5 epochs
#         if epoch % 5 == 0:
#             policy_net.eval()
#             val_losses = []
#             with torch.no_grad():
#                 for _ in range(10):
#                     vs, va, vr, vns, vd = val_memory.sample(batch_size)
#                     vs, va, vr, vns, vd = (vs.to(device), va.to(device), vr.to(device),
#                                             vns.to(device), vd.to(device))
#                     vq = policy_net(vs).gather(1, va)
#                     vt = vr + GAMMA * target_net(vns).max(dim=1, keepdim=True).values * (~vd).float()
#                     val_losses.append(criterion(vq, vt).item())
#             val_loss = np.mean(val_losses)
#             wr5, wr10, lr5, lr10 = recall_at_k(policy_net, val_df, k_values=[5, 10])

#             print(f'{epoch:>6} | {chunk_name:>14} | {avg_loss:>10.4f} | {val_loss:>9.4f} | '
#                   f'{wr5:>6.1f}% | {wr10:>7.1f}% | {lr5:>7.1f}% | {lr10:>8.1f}%')

#             if val_loss < best_val_loss:
#                 best_val_loss = val_loss
#                 patience      = 0
#                 torch.save(policy_net.state_dict(), MODEL_BEST)
#                 print(f'  ✓ Best model saved (val_loss={best_val_loss:.4f})')
#             else:
#                 patience += 1
#                 if patience >= MAX_PATIENCE:
#                     print(f'\nEarly stopping at epoch {epoch} — best val_loss={best_val_loss:.4f}')
#                     break

#             policy_net.train()

#     torch.save(policy_net.state_dict(), MODEL_FINAL)
#     print(f'\nSaved → Best  : {MODEL_BEST}')
#     print(f'Saved → Final : {MODEL_FINAL}')
#     return policy_net, target_net


# policy_net, target_net = train_chunked(val_transitions, CHUNK_DIR, num_chunks)

## 11. Final Evaluation on Test Set

In [71]:
# Load best checkpoint
policy_net.load_state_dict(torch.load(MODEL_BEST, map_location=device))
policy_net.eval()

print('Building test transitions...')
test_transitions = build_transitions(test_df, both_perspectives=True)

print('\n' + '=' * 70)
print('FINAL EVALUATION — HELD-OUT TEST SET')
print('=' * 70)
wr5, wr10, lr5, lr10 = recall_at_k(policy_net, test_df, k_values=[5, 10])
print(f'  Winning team — Recall@5: {wr5:.1f}%  |  Recall@10: {wr10:.1f}%')
print(f'  Losing team  — Recall@5: {lr5:.1f}%  |  Recall@10: {lr10:.1f}%')
print(f'\n  Random baseline → R@5 ≈ 3.8%  | R@10 ≈ 7.6%')
print(f'  Target (good)  → R@5 > 8–12% | R@10 > 15–20%')
print(f'\n  Win > Lose gap : {wr5 - lr5:.1f}% (positive = model favors winning picks)')

Building test transitions...
Processed: 23447 | Skipped: 0 | Transitions: 234470

FINAL EVALUATION — HELD-OUT TEST SET
  Winning team — Recall@5: 5.3%  |  Recall@10: 10.7%
  Losing team  — Recall@5: 4.3%  |  Recall@10: 9.8%

  Random baseline → R@5 ≈ 3.8%  | R@10 ≈ 7.6%
  Target (good)  → R@5 > 8–12% | R@10 > 15–20%

  Win > Lose gap : 1.0% (positive = model favors winning picks)


## 12. Recommendation Function

```python
recommend(
    ally_picks   = [{'hero': 'Lancelot', 'lane': 'Jungler'}, ...],
    enemy_picks  = [{'hero': 'Fanny',    'lane': 'Jungler'}, ...],
    banned       = ['Gloo', 'Sora', ...],
    player_lanes = ['Exp Laner', 'Gold Laner'],  # None = all lanes
    top_n        = 10
)
```

In [72]:
def recommend(ally_picks, enemy_picks, banned, player_lanes=None, top_n=10):
    ally_heroes  = [p['hero'] for p in ally_picks]
    enemy_heroes = [p['hero'] for p in enemy_picks]
    unavailable  = set(ally_heroes + enemy_heroes + banned)

    state = encode_state(ally_heroes, enemy_heroes).to(device)
    policy_net.eval()
    with torch.no_grad():
        q_values = policy_net(state.unsqueeze(0)).squeeze(0)

    all_scored = [
        (hero, q_values[idx].item())
        for hero, idx in hero_to_id.items()
        if hero not in unavailable
    ]

    def make_entry(h, s):
        return {'hero': h, 'score': round(s, 4),
                'roles': hero_roles.get(h, []),
                'specs': list(hero_specs.get(h, [])),
                'lane':  hero_lanes.get(h, [])}

    if player_lanes is None:
        return {'All': [make_entry(h, s) for h, s in sorted(all_scored, key=lambda x: x[1], reverse=True)[:top_n]]}

    results = {}
    for lane in player_lanes:
        lane_heroes = sorted(
            [(h, s) for h, s in all_scored if lane in hero_lanes.get(h, [])],
            key=lambda x: x[1], reverse=True
        )
        results[lane] = [make_entry(h, s) for h, s in lane_heroes[:top_n]]
    return results


def print_recommendations(results, banned):
    print(f'Banned: {banned}\n')
    for lane, heroes in results.items():
        print(f'  {lane}:')
        for rank, r in enumerate(heroes, 1):
            print(f'    {rank:2}. {r["hero"]:<20}  score={r["score"]:.4f}  {r["roles"]}')
        print()

## 13. Example Usage

In [76]:
# Example 1: Exp Laner + Gold Laner needed
print('=' * 60)
print('Example 1: Exp Laner + Gold Laner needed')
print('=' * 60)
banned1  = ['Hilda', 'Gusion', 'Sora', 'Gloo', 'Ixia', 'Floryn', 'Zetian', 'Zhuxin', 'Sora', 'Hilda']
results1 = recommend(
    ally_picks   = [{'hero': 'Hanzo', 'lane': 'Jungler'},
                    {'hero': 'Gord',   'lane': 'Mid Laner'},
                    {'hero': 'Freya',   'lane': 'Gold Laner'},
                    {'hero': 'Arlott',   'lane': 'Exp Laner'},],

    enemy_picks  = [{'hero': 'Saber', 'lane': 'Roamer'},
                    {'hero': 'Granger', 'lane': 'Gold Laner'},
                    {'hero': 'Selena',   'lane': 'Mid Laner'},
                    {'hero': 'Fredrinn',   'lane': 'Jungler'},
                    {'hero': 'Masha',   'lane': 'Exp Laner'},],
    banned       = banned1,
    player_lanes = ['Roamer'],
    top_n        = 10
)
print_recommendations(results1, banned1)


# Example 2: Jungler + Exp Laner needed
print('=' * 60)
print('Example 2: Jungler + Exp Laner needed')
print('=' * 60)
banned2  = getBannedHeroes()
results2 = recommend(
    ally_picks   = [{'hero': 'Gusion', 'lane': 'Mid Laner'},
                    {'hero': 'Atlas',  'lane': 'Roamer'}],

    enemy_picks  = [{'hero': 'Fanny',     'lane': 'Jungler'},
                    {'hero': 'Karrie',    'lane': 'Gold Laner'},
                    {'hero': 'Esmeralda', 'lane': 'Exp Laner'}],
    banned       = banned2,
    player_lanes = ['Jungler', 'Exp Laner'],
    top_n        = 10
)
print_recommendations(results2, banned2)


# Example 3: First pick — empty draft
print('=' * 60)
print('Example 3: First pick — no picks yet, Gold Laner')
print('=' * 60)
banned3  = getBannedHeroes()
results3 = recommend(
    ally_picks   = [],
    enemy_picks  = [],
    banned       = banned3,
    player_lanes = ['Gold Laner'],
    top_n        = 10
)
print_recommendations(results3, banned3)

Example 1: Exp Laner + Gold Laner needed
Banned: ['Hilda', 'Gusion', 'Sora', 'Gloo', 'Ixia', 'Floryn', 'Zetian', 'Zhuxin', 'Sora', 'Hilda']

  Roamer:
     1. Baxia                 score=0.3436  ['Tank']
     2. Chip                  score=0.3254  ['Support', 'Tank']
     3. Natalia               score=0.2902  ['Assassin']
     4. Minotaur              score=0.2491  ['Tank', 'Support']
     5. Mathilda              score=0.2213  ['Support', 'Assassin']
     6. Kaja                  score=0.2158  ['Fighter', 'Support']
     7. Kalea                 score=0.2051  ['Support', 'Fighter']
     8. Khaleed               score=0.1971  ['Fighter']
     9. Hylos                 score=0.1962  ['Tank']
    10. Faramis               score=0.1937  ['Support', 'Mage']

Example 2: Jungler + Exp Laner needed
Banned: ['Jawhead', 'Miya', 'Melissa', 'Balmond', 'Valir', 'Kagura', 'Arlott', 'Kadita', 'Lapu-Lapu', 'Alice']

  Jungler:
     1. Baxia                 score=2.0568  ['Tank']
     2. Suyou        